# Elastic Net (L1+L2) 로지스틱 회귀

로직은 [elastic_net.py](elastic_net.py), 이 노트북은 그걸 불러와서 실행 + 결과 해석을 남긴다.

**왜 시도했나**: CatBoost(트리)에서는 다중공선성 제거가 오히려 손해였다 (트리는 계수를 안 써서 안 불안정해지고, 파생 합계가 지름길 역할을 함 — `baseline_catboost.py` 참고). 선형모델은 정반대로 상관 높은 변수를 그대로 두면 계수 추정이 불안정해지니, 여기서는 완전 중복 변수를 제거하는 게 맞다. 이 원칙을 실제로 적용해볼 겸, L0(무규제)/L1(Lasso)/L1+L2(Elastic Net) 페널티 차이도 비교해본다.

**결론 미리 요약**: 첫 시도는 완전히 실패했다(베이스라인보다 나쁨, 실제 LB 342.74). 원인을 진단해서 고치니(`game_type × season_regime` 교호작용 추가) 0점대에서 336점대로 올라갔지만, 여전히 CatBoost(로컬 734.49 / 실제 LB 789.23)에는 크게 못 미친다. 실제 제출용이 아니라 "왜 선형모델이 이 데이터에서 트리보다 불리한가"를 확인하는 기록.

In [1]:
from elastic_net import *  # noqa: F403
import warnings
warnings.filterwarnings("ignore")

df = prepare(load("train.csv"))
print(f"수치형 {len(NUMERIC_FEATURES)}개(중복 3개 제거) + 범주형/이진/결측플래그 {len(CATEGORICAL_FEATURES) + len(MISSING_FLAGS)}개")

수치형 29개(중복 3개 제거) + 범주형/이진/결측플래그 15개


## 1. 전처리

CatBoost와 달리 선형모델은 결측치/범주형을 네이티브로 처리 못 해서 준비가 더 필요하다.

- **다중공선성 제거**: `run_total_before`(=run_top+run_bot), `num_runners_on`(=세 주자플래그 합), `away_win_expectancy`(=100-home) 3개 삭제. `r=+1`뿐 아니라 `r=-1`도 완전 중복으로 취급.
- **결측치**: 16개 결측 컬럼을 원인 3그룹 플래그(`is_pitcher_coldstart`, `is_batter_coldstart`, `is_missing_recent_games`)로 압축 + 중앙값 대체. 중앙값은 **sklearn Pipeline 안(SimpleImputer)에서 계산**해서 train/valid 분할 후에도 검증 데이터 정보가 새지 않게 함 (첫 버전엔 분할 전에 미리 fillna하는 버그가 있었음 — 결측이 희귀해서 영향은 작았겠지만 원칙적으로 틀렸었음, 지금은 고침).
- **범주형**: `OneHotEncoder(drop="if_binary")` — 이진(2개 값)은 더미변수 함정(완전공선성) 방지를 위해 1개 컬럼으로, 3개 이상인 것(`base_state`, team_id, month, dayofweek)은 전체 원핫.
- **스케일링**: `StandardScaler`. 계수 크기 비교 가능하게 하고 saga solver 수렴 속도 개선.

## 2. 1차 시도: 페널티 비교 (교호작용 없이)

`game_type`을 그냥 단독 변수로 넣은 상태에서 L0(무규제)/L1(Lasso)/L1+L2(Elastic Net)를 비교.

In [2]:
# game_type_regime 교호작용을 추가하기 전, 실제로 돌렸던 결과를 그대로 저장해둔 파일
cmp_before = pd.read_csv("penalty_comparison.csv", index_col=0)
cmp_before

,n,r,brier,score,auc,nonzero_coef
variant,,,,,,
none,253507,0.4861,0.250030,0.0,0.5249,86/86
l1,253507,0.4861,0.250031,0.0,0.5249,80/86
elasticnet,253507,0.4861,0.250032,0.0,0.5249,80/86


**셋 다 베이스라인보다 나빴다** (Brier > r(1-r)=0.249807, score가 0으로 깎임). 페널티 종류(규제 강도/L1 비율)를 바꿔도 전혀 차이가 없었다 — 이건 "규제가 부족/과함"의 문제가 아니라 **모델의 표현력 자체가 부족**하다는 신호다. `game_type`의 성공률이 2019~2022엔 높다가(0.59~0.71) 2023부터 뒤집히는데(0.46~0.47, [eda2.ipynb](../eda/eda2.ipynb) 참고), 선형모델은 이 변수에 계수를 딱 하나만 배정할 수 있어서 두 시기의 상반된 패턴을 하나의 숫자로 뭉갤 수밖에 없다. 실제 제출도 해봤음: **Public LB 342.74** (로컬 진단과 방향 일치, 수료 기준 549.51도 못 넘김 — 이 모델은 최종 제출로 쓰면 안 됨).

## 3. 진단 후 수정: `game_type × season_regime` 교호작용 추가

`game_type`을 단독으로 넣는 대신, `season≥2023` 여부와 합친 조합 카테고리로 바꿨다 — `R_pre2023`/`R_post2023`/`F_pre2023`/`F_post2023` 4개 값.  실제 효과를 봤던 E4와 같은 발상 (그 팀도 `game_type_F × post_regime` 교호작용에서 최대 단일 개선을 얻었음). 핵심은 **따로따로 두 변수로 넣는 것과 다르다** — 조합 카테고리는 "F이면서 2023 이후"에 대해 별도 계수를 직접 배우게 해준다.

In [3]:
print("game_type_regime 값:", sorted(df["game_type_regime"].unique()))
cmp_after = pd.read_csv("penalty_comparison_with_interaction.csv", index_col=0)
cmp_after

game_type_regime 값: ['F_post2023', 'F_pre2023', 'R_post2023', 'R_pre2023']


,n,r,brier,score,auc,nonzero_coef
variant,,,,,,
none,253507,0.4861,0.248968,335.99,0.5346,89/89
l1,253507,0.4861,0.248967,336.38,0.5346,85/89
elasticnet,253507,0.4861,0.248967,336.24,0.5346,85/89


**셋 다 336점 근처로 크게 개선됐다** (0 → 336, Brier 0.25003 → 0.24897, AUC 0.5249 → 0.5346). 페널티 종류는 여전히 거의 차이가 없다(335.99/336.38/336.24) — "규제 강도는 안 중요하고 모델 형태(교호작용 유무)가 중요하다"는 진단이 실측으로 확인된 것.

다만 여전히 CatBoost(로컬 734.49 / 실제 LB 789.23)에는 크게 못 미친다. 우리가 **직접 찾아서 손으로 넣어준 교호작용은 이거 하나뿐**인데, 실제로는 더 있을 수 있는 교호작용을 트리는 데이터로부터 자동으로 다 찾아내는 반면, 선형모델은 찾아서 넣어준 것만큼만 좋아진다 — 이게 이 데이터에서 선형모델이 트리보다 근본적으로 불리한 이유다.

## 4. 최종 모델 (참고용, 제출 비추천)

L1+L2(Elastic Net, l1_ratio=0.5)로 2019~2024 전체 재학습. 위에서 확인했듯 CatBoost보다 확실히 낮으니 실제 제출은 CatBoost 쪽을 쓴다.

In [4]:
final_pipe = train(df, penalty="elasticnet", l1_ratio=0.5)

MODEL_DIR.mkdir(exist_ok=True)
import joblib
model_path = MODEL_DIR / "elastic_net.joblib"
joblib.dump(final_pipe, model_path)
print("saved", model_path)

saved /Users/choehabin/Library/CloudStorage/OneDrive-개인/lg aimers/모델링/modeling/models/elastic_net.joblib


## 결론

1. **선형모델 자체가 이 데이터와 안 맞는 건 아니다** — 대부분의 `asof_*` 변수는 타겟과 거의 선형(단조) 관계라 선형모델이 잘 다룬다.
2. **`game_type` 하나가 문제였다** — 효과의 부호가 2023을 기점으로 뒤집히는 변수라, 계수 하나만 배정하는 선형모델의 기본 가정을 깬다.
3. **페널티 종류(L0/L1/L1+L2)는 이 문제의 해결책이 아니었다** — 규제를 아무리 바꿔도 표현력 부족은 못 고친다. 교호작용 변수를 명시적으로 만들어줘야 했다.
4. **교호작용을 고치니 확실히 좋아졌지만(0→336) 여전히 트리에는 못 미친다** — 트리는 이런 교호작용을 자동으로 찾는데, 선형모델은 사람이 찾아서 넣어준 것만큼만 좋아지기 때문. 이 데이터엔 우리가 못 찾은 다른 교호작용이 더 있을 가능성이 높다.

**최종 판단**: 실제 제출은 CatBoost(로컬 734.49 / 실제 LB 789.23)를 유지. Elastic Net은 "왜 트리가 유리한가"를 실증적으로 보여준 학습 기록으로 남긴다.